# 08 — Model Evaluation

This notebook evaluates the projection system in three ways:

1. Historical walk forward validation of the team strength baseline.
2. Comparison of 2026 game predictions against preseason betting market expectations.
3. Validation of the final representative season standings output.

The betting market is treated only as an external benchmark and is never used as an input to the projection model.

In [9]:
from pathlib import Path

import numpy as np
import pandas as pd
import polars as pl
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error

In [10]:
PROJECT_ROOT = Path("..")
DATA_DIR = PROJECT_ROOT / "data"
PROCESSED_DIR = DATA_DIR / "processed"

In [11]:
historical_team_features = pl.read_parquet(
    PROCESSED_DIR / "historical_team_features.parquet"
)

game_predictions_2026 = pd.read_parquet(
    PROCESSED_DIR / "2026_game_predictions.parquet"
)

season_projections_2026 = pd.read_parquet(
    PROCESSED_DIR / "2026_season_projections.parquet"
)

predicted_standings_2026 = pd.read_parquet(
    PROCESSED_DIR / "2026_predicted_standings.parquet"
)

## Historical Team-Strength Validation

The 2026 season has not yet been played, so final 2026 accuracy cannot be measured yet.

Instead, the historical baseline used in Notebook 05 is evaluated with walk forward validation.

For each historical team season, point differential per game from the previous three seasons is combined using the same weighting used for the 2026 projection:

- 60% previous season
- 25% two seasons ago
- 15% three seasons ago

A linear regression is then trained only on seasons that occurred before each validation season and used to predict that season's actual point differential per game.

This keeps the evaluation leakage safe and ensures that the historical test matches the baseline methodology currently used in the 2026 model.

In [12]:
team_history = (
    historical_team_features
    .select([
        "season",
        "team",
        "point_diff_per_game"
    ])
    .to_pandas()
    .sort_values([
        "team",
        "season"
    ])
)

team_history["pd_lag1"] = (
    team_history
    .groupby("team")["point_diff_per_game"]
    .shift(1)
)

team_history["pd_lag2"] = (
    team_history
    .groupby("team")["point_diff_per_game"]
    .shift(2)
)

team_history["pd_lag3"] = (
    team_history
    .groupby("team")["point_diff_per_game"]
    .shift(3)
)

team_history["weighted_prior_pd"] = (
    0.60 * team_history["pd_lag1"]
    + 0.25 * team_history["pd_lag2"]
    + 0.15 * team_history["pd_lag3"]
)

historical_validation = (
    team_history
    .dropna(
        subset=[
            "weighted_prior_pd",
            "point_diff_per_game"
        ]
    )
    .copy()
)

In [13]:
validation_rows = []

validation_seasons = sorted(
    season
    for season in historical_validation["season"].unique()
    if 2020 <= season <= 2025
)

for season in validation_seasons:
    train = historical_validation[
        historical_validation["season"] < season
    ]

    test = historical_validation[
        historical_validation["season"] == season
    ]

    if train.empty or test.empty:
        continue

    model = LinearRegression()

    model.fit(
        train[["weighted_prior_pd"]],
        train["point_diff_per_game"]
    )

    predictions = model.predict(
        test[["weighted_prior_pd"]]
    )

    validation_rows.append({
        "season": season,
        "mae": mean_absolute_error(
            test["point_diff_per_game"],
            predictions
        ),
        "correlation": test[
            "point_diff_per_game"
        ].corr(
            pd.Series(
                predictions,
                index=test.index
            )
        )
    })

historical_validation_results = pd.DataFrame(
    validation_rows
)

In [14]:
print(
    historical_validation_results
    .round(3)
    .to_string(index=False)
)

print()

print(
    "Average MAE:",
    round(
        historical_validation_results["mae"].mean(),
        3
    )
)

print(
    "Average correlation:",
    round(
        historical_validation_results["correlation"].mean(),
        3
    )
)

 season   mae  correlation
   2020 4.965        0.401
   2021 4.492        0.585
   2022 4.250        0.276
   2023 3.846        0.564
   2024 5.242        0.291
   2025 4.445        0.356

Average MAE: 4.54
Average correlation: 0.412


## 2026 Betting-Market Benchmark

Betting markets provide a useful external benchmark because market prices incorporate a large amount of public information and expectation.

The model's 2026 predictions remain fully independent of betting-market data. Market information is introduced only here to identify where the model agrees or disagrees with consensus expectations.

In [15]:
import nflreadpy as nfl

schedule_2026 = nfl.load_schedules(
    seasons=[2026]
)

schedule_2026 = schedule_2026.filter(
    pl.col("game_type") == "REG"
)

## Model vs. Betting Market

The available preseason betting lines are compared directly with the model's expected home margin.

A positive margin indicates the home team is favored, while a negative margin indicates the away team is favored. Because betting lines are currently available for only part of the 2026 schedule, this comparison uses only games with an available market spread.

In [16]:
market_spreads = (
    schedule_2026
    .filter(
        pl.col("spread_line").is_not_null()
    )
    .select([
        "game_id",
        "spread_line"
    ])
    .to_pandas()
)

market_comparison = (
    game_predictions_2026
    .merge(
        market_spreads,
        on="game_id",
        how="inner"
    )
)

market_comparison["model_market_difference"] = (
    market_comparison["expected_home_margin"]
    - market_comparison["spread_line"]
)

market_comparison["absolute_difference"] = (
    market_comparison[
        "model_market_difference"
    ].abs()
)

In [17]:
market_mae = (
    market_comparison[
        "absolute_difference"
    ].mean()
)

market_correlation = (
    market_comparison[
        [
            "expected_home_margin",
            "spread_line"
        ]
    ]
    .corr()
    .iloc[0, 1]
)

market_comparison["model_favorite"] = np.where(
    market_comparison[
        "expected_home_margin"
    ] >= 0,
    market_comparison["home_team"],
    market_comparison["away_team"]
)

market_comparison["market_favorite"] = np.where(
    market_comparison[
        "spread_line"
    ] >= 0,
    market_comparison["home_team"],
    market_comparison["away_team"]
)

favorite_agreement = (
    market_comparison["model_favorite"]
    == market_comparison["market_favorite"]
).mean()

In [18]:
print(
    "Games with market lines:",
    len(market_comparison)
)

print(
    "Average absolute difference:",
    round(market_mae, 3),
    "points"
)

print(
    "Model-market correlation:",
    round(market_correlation, 3)
)

print(
    "Favorite agreement:",
    f"{favorite_agreement:.1%}"
)

Games with market lines: 100
Average absolute difference: 2.045 points
Model-market correlation: 0.856
Favorite agreement: 81.0%


### Largest Market Disagreements

Individual differences from betting-market lines are expected because market data is not used as an input to the model.

The largest disagreements are reviewed as a diagnostic rather than treated automatically as model errors.

In [19]:
largest_market_disagreements = (
    market_comparison
    .sort_values(
        "absolute_difference",
        ascending=False
    )
    [
        [
            "week",
            "away_team",
            "home_team",
            "expected_home_margin",
            "spread_line",
            "model_market_difference"
        ]
    ]
    .head(10)
    .reset_index(drop=True)
)

largest_market_disagreements

,week,away_team,home_team,expected_home_margin,spread_line,model_market_difference
0,5,CIN,MIA,0.059707,-6.0,6.059707
1,6,TEN,IND,9.043098,3.5,5.543098
2,3,KC,MIA,-1.981971,-7.5,5.518029
3,7,MIA,NYJ,-2.993907,2.5,-5.493907
4,4,ARI,NYG,1.985682,7.0,-5.014318
5,1,MIA,LV,-1.486560,3.5,-4.986560
6,3,CIN,PIT,3.046864,-1.5,4.546864
7,5,HOU,TEN,-7.793949,-3.5,-4.293949
8,2,IND,KC,2.352551,6.5,-4.147449
9,4,IND,WAS,-2.534095,1.5,-4.034095


## Comparison with a Simple Baseline

The historical team strength model is also compared with a simple benchmark that uses each team's previous season point differential as its prediction for the following season.

Both approaches are evaluated over the same 2020–2025 walk forward validation period. This tests whether the weighted three year baseline adds predictive value beyond simply carrying forward the previous season.

In [20]:
benchmark_rows = []

for season in validation_seasons:
    train = historical_validation[
        historical_validation["season"] < season
    ]

    test = historical_validation[
        historical_validation["season"] == season
    ]

    if train.empty or test.empty:
        continue

    model = LinearRegression()

    model.fit(
        train[["weighted_prior_pd"]],
        train["point_diff_per_game"]
    )

    model_predictions = model.predict(
        test[["weighted_prior_pd"]]
    )

    naive_predictions = (
        test["pd_lag1"]
        .to_numpy()
    )

    benchmark_rows.append({
        "season": season,
        "model_mae": mean_absolute_error(
            test["point_diff_per_game"],
            model_predictions
        ),
        "naive_mae": mean_absolute_error(
            test["point_diff_per_game"],
            naive_predictions
        )
    })

baseline_comparison = pd.DataFrame(
    benchmark_rows
)

In [21]:
print(
    baseline_comparison
    .round(3)
    .to_string(index=False)
)

print()

print(
    "Model average MAE:",
    round(
        baseline_comparison["model_mae"].mean(),
        3
    )
)

print(
    "Naive average MAE:",
    round(
        baseline_comparison["naive_mae"].mean(),
        3
    )
)

 season  model_mae  naive_mae
   2020      4.965      5.746
   2021      4.492      4.746
   2022      4.250      5.164
   2023      3.846      4.457
   2024      5.242      5.794
   2025      4.445      6.254

Model average MAE: 4.54
Naive average MAE: 5.36


### Baseline Comparison Results

The weighted three year model outperforms the previous-season benchmark in every validation season from 2020 through 2025.

This provides evidence that the historical baseline adds predictive value beyond simply carrying forward the most recent season.

This validation applies specifically to the historical team strength baseline. The 2026 personnel and roster continuity adjustments cannot be evaluated identically because equivalent historical versions of those inputs are not available.

## Model vs. Market Diagnostics

Individual disagreements with the betting market are not necessarily model errors. An independent projection model should disagree with the market in some games.

The more important diagnostic is whether those disagreements are systematically concentrated around the same teams or consistently occur in the same direction.

This section measures the overall size and direction of model market differences and identifies teams that the model may systematically value differently from the market.

In [22]:
market_eval = market_comparison.copy()

market_eval["absolute_market_difference"] = (
    market_eval["model_market_difference"].abs()
)

print("MODEL VS MARKET SUMMARY")
print("-" * 40)

print(
    f"Games evaluated: "
    f"{len(market_eval)}"
)

print(
    f"Mean difference: "
    f"{market_eval['model_market_difference'].mean():.3f}"
)

print(
    f"Median difference: "
    f"{market_eval['model_market_difference'].median():.3f}"
)

print(
    f"Mean absolute difference: "
    f"{market_eval['absolute_market_difference'].mean():.3f}"
)

print(
    f"RMSE: "
    f"{np.sqrt(np.mean(market_eval['model_market_difference'] ** 2)):.3f}"
)

print(
    f"Std. deviation of differences: "
    f"{market_eval['model_market_difference'].std():.3f}"
)

MODEL VS MARKET SUMMARY
----------------------------------------
Games evaluated: 100
Mean difference: 0.009
Median difference: 0.275
Mean absolute difference: 2.045
RMSE: 2.499
Std. deviation of differences: 2.511


In [23]:
difference_thresholds = [1, 2, 3, 4, 5]

threshold_results = []

for threshold in difference_thresholds:
    count = (
        market_eval["absolute_market_difference"]
        >= threshold
    ).sum()

    threshold_results.append({
        "threshold": threshold,
        "games": count,
        "percentage": (
            count / len(market_eval) * 100
        )
    })

threshold_results = pd.DataFrame(
    threshold_results
)

threshold_results.round(2)

,threshold,games,percentage
0,1,71,71.0
1,2,44,44.0
2,3,26,26.0
3,4,10,10.0
4,5,5,5.0


In [24]:
home_team_bias = market_eval[
    [
        "week",
        "home_team",
        "model_market_difference"
    ]
].copy()

home_team_bias = home_team_bias.rename(
    columns={
        "home_team": "team",
        "model_market_difference": "team_market_difference"
    }
)


away_team_bias = market_eval[
    [
        "week",
        "away_team",
        "model_market_difference"
    ]
].copy()

away_team_bias = away_team_bias.rename(
    columns={
        "away_team": "team",
        "model_market_difference": "team_market_difference"
    }
)

away_team_bias["team_market_difference"] *= -1


team_market_bias = pd.concat(
    [
        home_team_bias,
        away_team_bias
    ],
    ignore_index=True
)

In [25]:
team_bias_summary = (
    team_market_bias
    .groupby("team")
    .agg(
        games=("team_market_difference", "count"),
        average_bias=("team_market_difference", "mean"),
        median_bias=("team_market_difference", "median"),
        average_absolute_bias=(
            "team_market_difference",
            lambda x: x.abs().mean()
        ),
        max_absolute_bias=(
            "team_market_difference",
            lambda x: x.abs().max()
        )
    )
    .reset_index()
)

team_bias_summary["same_direction_rate"] = (
    team_market_bias
    .assign(
        positive=lambda x:
            x["team_market_difference"] > 0
    )
    .groupby("team")["positive"]
    .apply(
        lambda x:
            max(
                x.mean(),
                1 - x.mean()
            )
    )
    .values
)

team_bias_summary = (
    team_bias_summary
    .sort_values(
        "average_absolute_bias",
        ascending=False
    )
    .reset_index(drop=True)
)

team_bias_summary.round(3)

,team,games,average_bias,median_bias,average_absolute_bias,max_absolute_bias,same_direction_rate
0,MIA,6,4.710,5.240,4.710,6.060,1.000
1,ARI,6,3.483,3.312,3.483,5.014,1.000
2,CIN,6,-3.416,-3.133,3.416,6.060,1.000
3,TEN,6,-2.880,-2.510,2.880,5.543,1.000
4,KC,5,-2.403,-3.004,2.772,5.518,0.800
5,DEN,6,2.666,3.200,2.666,3.752,1.000
6,IND,7,2.449,2.722,2.584,5.543,0.857
7,HOU,7,2.138,2.530,2.419,4.294,0.857
8,NYG,7,-1.336,-1.718,2.356,5.014,0.714
9,CAR,6,-2.245,-2.668,2.339,3.788,0.833


In [26]:
systematic_bias_flags = (
    team_bias_summary[
        (
            team_bias_summary["games"] >= 3
        )
        &
        (
            team_bias_summary["average_bias"].abs()
            >= 2.0
        )
        &
        (
            team_bias_summary["same_direction_rate"]
            >= 0.75
        )
    ]
    .copy()
)

print(
    "TEAMS WITH CONSISTENT MODEL-MARKET DISAGREEMENT"
)

if systematic_bias_flags.empty:
    print(
        "No teams currently meet the systematic-bias criteria."
    )
else:
    display(
        systematic_bias_flags.round(3)
    )

TEAMS WITH CONSISTENT MODEL-MARKET DISAGREEMENT


,team,games,average_bias,median_bias,average_absolute_bias,max_absolute_bias,same_direction_rate
0,MIA,6,4.710,5.240,4.710,6.060,1.000
1,ARI,6,3.483,3.312,3.483,5.014,1.000
2,CIN,6,-3.416,-3.133,3.416,6.060,1.000
3,TEN,6,-2.880,-2.510,2.880,5.543,1.000
4,KC,5,-2.403,-3.004,2.772,5.518,0.800
5,DEN,6,2.666,3.200,2.666,3.752,1.000
6,IND,7,2.449,2.722,2.584,5.543,0.857
7,HOU,7,2.138,2.530,2.419,4.294,0.857
9,CAR,6,-2.245,-2.668,2.339,3.788,0.833
10,LAC,6,-2.171,-2.682,2.171,3.395,1.000


## Final Standings Validation

The official 2026 standings prediction is selected from the Monte Carlo simulations rather than created by rounding each team's average projected wins.

The selected simulation is required to have league wide record dispersion within the range observed from 2020–2025 and is then chosen as the realistic simulation closest overall to the model's expected team win totals.

The checks below confirm that the final prediction represents one internally consistent NFL season.

In [27]:
final_standings_sd = (
    predicted_standings_2026[
        "predicted_wins"
    ].std()
)

total_predicted_wins = (
    predicted_standings_2026[
        "predicted_wins"
    ].sum()
)

total_predicted_losses = (
    predicted_standings_2026[
        "predicted_losses"
    ].sum()
)

print(
    "Total predicted wins:",
    total_predicted_wins
)

print(
    "Total predicted losses:",
    total_predicted_losses
)

print(
    "Final standings SD:",
    round(
        final_standings_sd,
        3
    )
)

print(
    "Best predicted record:",
    int(
        predicted_standings_2026[
            "predicted_wins"
        ].max()
    ),
    "-",
    int(
        predicted_standings_2026.loc[
            predicted_standings_2026[
                "predicted_wins"
            ].idxmax(),
            "predicted_losses"
        ]
    )
)

print(
    "Worst predicted record:",
    int(
        predicted_standings_2026[
            "predicted_wins"
        ].min()
    ),
    "-",
    int(
        predicted_standings_2026.loc[
            predicted_standings_2026[
                "predicted_wins"
            ].idxmin(),
            "predicted_losses"
        ]
    )
)

Total predicted wins: 272
Total predicted losses: 272
Final standings SD: 2.782
Best predicted record: 15 - 2
Worst predicted record: 3 - 14


In [28]:
historical_validation_results.to_parquet(
    PROCESSED_DIR / "historical_model_validation.parquet",
    index=False
)

baseline_comparison.to_parquet(
    PROCESSED_DIR / "historical_baseline_comparison.parquet",
    index=False
)

market_comparison.to_parquet(
    PROCESSED_DIR / "2026_market_comparison.parquet",
    index=False
)

In [29]:
historical_model_mae = (
    historical_validation_results[
        "mae"
    ].mean()
)

naive_baseline_mae = (
    baseline_comparison[
        "naive_mae"
    ].mean()
)

historical_improvement = (
    1
    - historical_model_mae
    / naive_baseline_mae
)

print("FINAL MODEL EVALUATION")
print("-" * 40)

print(
    "Historical seasons evaluated:",
    len(historical_validation_results)
)

print(
    "Historical model MAE:",
    round(
        historical_model_mae,
        3
    )
)

print(
    "Naive baseline MAE:",
    round(
        naive_baseline_mae,
        3
    )
)

print(
    "Improvement over naive baseline:",
    f"{historical_improvement:.1%}"
)

print()

print(
    "2026 market games evaluated:",
    len(market_comparison)
)

print(
    "Model-market correlation:",
    round(
        market_correlation,
        3
    )
)

print(
    "Average absolute market difference:",
    round(
        market_mae,
        3
    )
)

print(
    "Favorite agreement:",
    f"{favorite_agreement:.1%}"
)

print(
    "Average signed market difference:",
    round(
        market_comparison[
            "model_market_difference"
        ].mean(),
        3
    )
)

print()

print(
    "Final standings SD:",
    round(
        final_standings_sd,
        3
    )
)

FINAL MODEL EVALUATION
----------------------------------------
Historical seasons evaluated: 6
Historical model MAE: 4.54
Naive baseline MAE: 5.36
Improvement over naive baseline: 15.3%

2026 market games evaluated: 100
Model-market correlation: 0.856
Average absolute market difference: 2.045
Favorite agreement: 81.0%
Average signed market difference: 0.009

Final standings SD: 2.782


## Evaluation Summary

The projection system was evaluated using historical out of sample testing, comparison with a simple historical benchmark, and an independent comparison with 2026 betting market lines.

### Historical Validation

The historical team strength baseline was evaluated using walk forward testing from 2020 through 2025. The weighted three year approach outperformed a simple previous season point differential benchmark in every validation season, providing evidence that the baseline contains useful predictive signal beyond the most recent season alone.

### 2026 Market Comparison

Betting market lines are used only as an external benchmark and are never included as model inputs. The model shows strong overall agreement with market expectations while still producing meaningful independent disagreements.

Team level diagnostics are also used to identify cases where the model consistently values a team differently from the market. These disagreements are treated as areas for review rather than automatically being considered model errors.

### Final Standings

The official standings prediction represents one complete simulated season rather than the average outcome across all simulations. The representative season is selected from historically realistic simulations based on its overall consistency with the model's expected team win totals.

This produces exact and internally consistent win-loss records while preserving the team-strength estimates, schedule effects, game probabilities, and Monte Carlo framework developed throughout the project.

The complete 2026 model can be evaluated against actual game and season outcomes once the season has been played.